# Thêm Thư Viện

In [2]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [3]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=dwh_library;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2024;'
)

## Đọc data từ SQL Server

In [4]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Tenform = """SELECT ID, 
                        dbo.DecodeUTF8String(Ten_form) AS Ten_form, 
                        Nguoi_tao,
                        Ngay_tao,
                        Ngay_sua_cuoi
                        FROM Ten_Form
                        """
df_tenform = pd.read_sql(query_Tenform, conn_libol)
print(df_tenform)

    ID                         Ten_form      Nguoi_tao            Ngay_tao  \
0   37                    Sách (USMARC)  Administrator 2001-05-19 16:00:51   
1   38              Băng Video (USMARC)  Administrator 2001-05-31 16:58:22   
2   40                Âm thanh (USMARC)  Administrator 2001-06-02 08:52:53   
3   39            Tệp máy tính (USMARC)  Administrator 2001-05-31 17:34:27   
4   41                  Bản đồ (USMARC)  Administrator 2001-08-09 17:20:27   
5   42         Ấn phẩm định kỳ (USMARC)  Administrator 2001-08-10 08:46:06   
6   43                   Sách (rút gọn)  Administrator 2001-08-21 11:50:25   
7   81  Biên mục Tiêu chuẩn và Quy phạm         DHSPKT 2002-08-21 08:55:07   
8   67                  Bài báo(DHSPKT)  Administrator 2001-11-27 11:09:01   
9   68                        Bài trích  Administrator 2001-11-27 11:16:28   
10  85               Biên mục Bài trích  Administrator 2002-08-21 12:36:16   
11  82                    Biên mục Sách         DHSPKT 2002-08-2

C:\Users\admin\AppData\Local\Temp\ipykernel_15708\64601752.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tenform = pd.read_sql(query_Tenform, conn_libol)


## Xử lý data

In [5]:
df_tenform['Ngay_tao'] = df_tenform['Ngay_tao'].dt.strftime('%Y%m%d').astype(int)
df_tenform['Ngay_sua_cuoi'] = df_tenform['Ngay_sua_cuoi'].dt.strftime('%Y%m%d').astype(int)

new_row = pd.DataFrame({'ID': [0], # Tạo hàng dữ liệu giả lập cho form không xác định
                        'Ten_form': ['(Không xác định)'],
                        'Nguoi_tao': ['(Không xác định)'],
                        'Ngay_tao': [0],
                        'Ngay_sua_cuoi': [0]})
df_tenform = pd.concat([df_tenform, new_row], ignore_index=True) # Thêm vào dataframe

df_tenform = df_tenform.sort_values(by="ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_tenform)

    ID                         Ten_form         Nguoi_tao  Ngay_tao  \
0    0                 (Không xác định)  (Không xác định)         0   
1   37                    Sách (USMARC)     Administrator  20010519   
2   38              Băng Video (USMARC)     Administrator  20010531   
3   39            Tệp máy tính (USMARC)     Administrator  20010531   
4   40                Âm thanh (USMARC)     Administrator  20010602   
5   41                  Bản đồ (USMARC)     Administrator  20010809   
6   42         Ấn phẩm định kỳ (USMARC)     Administrator  20010810   
7   43                   Sách (rút gọn)     Administrator  20010821   
8   67                  Bài báo(DHSPKT)     Administrator  20011127   
9   68                        Bài trích     Administrator  20011127   
10  81  Biên mục Tiêu chuẩn và Quy phạm            DHSPKT  20020821   
11  82                    Biên mục Sách            DHSPKT  20020821   
12  83          Biên mục Báo và Tạp chí            DHSPKT  20020821   
13  84

## Load data

### [Nếu cần] Clear bảng

In [6]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM DIM_Ten_form"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [7]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO DIM_Ten_form (ID_form, Ten_form, Nguoi_tao, Ngay_tao, Ngay_sua_cuoi) 
                VALUES (?, ?, ?, ?, ?)
               """
for index, row in df_tenform.iterrows():
    values = (row['ID'], 
                row['Ten_form'],
                row['Nguoi_tao'],
                row['Ngay_tao'],
                row['Ngay_sua_cuoi'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()